# Distillation Preprocessing (1)

Things we do:
1) Extract game states
2) Filter and permute game states

In [1]:
import dotenv

dotenv.load_dotenv()

True

In [2]:
from core.types import *
from doom.preprocessing.doom_game_state_perturbator import DoomGameStatePerturbator
from doom.preprocessing.doom_game_state_clusterer import DoomGameStateClusterer
from doom.utils.doom_game_state import DoomGameState
from sklearn.cluster import DBSCAN
from typing import Iterable
from pathlib import Path

import os
import numpy as np

In [3]:
%load_ext autoreload
%autoreload 2

## 1) Loading Game States

In [4]:
# Extract Game States from Logs

raw_game_states = [
    DoomGameState.model_validate_json(line[15:])
    for filepath in os.scandir("data/gamelogs")
    for line in open(filepath)
    if line.startswith("[GS] GAMESTATE ")
]
game_states = [
    GameStateEntry(
        id=f"state-{idx}",
        state=state
    )
    for idx, state in enumerate(raw_game_states)
]

print(f"Extracted {len(game_states)} game states from gameplay")

Extracted 58821 game states from gameplay


## 2) Preprocessing the Game States

In [5]:
# Eliminate uninteresting game states

def is_interesting(gs: DoomGameState) -> bool:
    if len(gs.MONSTERS) > 0: return True
    if gs.AIMED_AT.interactable: return True
    return False


game_states = [
    gse
    for gse in game_states
    if is_interesting(gse.state)
]

print(f"Reduced to {len(game_states)} interesting game states")

Reduced to 34081 interesting game states


In [6]:
# Only keep unique states (using clustering)

def to_feature_vector(gs: DoomGameState) -> np.ndarray:
    return DoomGameStateClusterer.to_feature_vector(gs)


features = np.array([to_feature_vector(gse.state) for gse in game_states])

clustering = DBSCAN(
    eps=1e-2,
    min_samples=1,
    metric="euclidean",
)
labels = clustering.fit_predict(features)

new_dataset = list()
for cluster_id in set(labels):
    if cluster_id == -1: continue

    # Get elements of the given cluster
    cluster_indices = np.where(labels == cluster_id)[0]
    cluster_features = features[cluster_indices]

    # Find the closest item to the cluster center
    centroid = cluster_features.mean(axis=0)
    distances = np.linalg.norm(cluster_features - centroid, axis=1)
    center_idx = cluster_indices[np.argmin(distances)]

    new_dataset.append(game_states[center_idx])

game_states = new_dataset

print(f"Reduced to {len(game_states)} cluster-center game states")

Reduced to 281 cluster-center game states


In [7]:
# Generate perturbations of the game states

def yield_perturbations(gs: DoomGameState) -> Iterable[DoomGameState]:
    return DoomGameStatePerturbator.perturbate(gs)


game_states = [
    GameStateEntry(
        id=f'{gse.id}-p{idx}',
        state=new_gs
    )
    for gse in game_states
    for idx, new_gs in enumerate(yield_perturbations(gse.state))
]

print(f"Applied perturbations and went up to {len(game_states)} game states")

Applied perturbations and went up to 960 game states


In [ ]:
# Save the current dataset
GameStateEntry.save_states(
    x=game_states,
    path=Path("data/gamestates/perturbated-gamestates.json")
)